# Opdracht ML
## Inleiding
https://www.kaggle.com/competitions/car-breakdown-prediction
data zit in train_...

Your goal is to use these features to classify each vehicle into one of two categories:

0 → No breakdown expected in the next 30 days
1 → Breakdown expected within the next 30 days

Like this
id,breakdown_next_30_days
1356,1
889,1

In [ ]:
# get the file path
file = "./src/train_CarBreakDown.csv"

In [ ]:
# breakdown class
class Breakdowns:
    def __init__(self, id: int, will_breakdown: bool):
        self.id = id
        self.will_breakdown = will_breakdown

In [ ]:
# get file data function
def get_car_breakdown_data(file: str):
    with open(file, "r", encoding="UTF-8") as f:
        data = f.readlines()
        return data

In [ ]:
# test read the data
dirty_data = get_car_breakdown_data(file)
print(dirty_data[:2])

['id,vehicle_brand,vehicle_age_years,mileage_km,engine_hours,last_service_km_ago,oil_quality_pct,avg_trip_length_km,weather_exposure,fuel_type,cleanliness_score,driver_satisfaction_score,tyre_type,breakdown_next_30_days\n', '959,Toyota,20.0,494744.9727215044,9700.481467070696,1672.0484451073842,61.66455595738916,30.36209049530085,high,petrol,68.11783295813919,8.559011249392881,,0\n']


In [ ]:
# panda dataframe import
import pandas as pd





# ================================
# CREATE DATAFRAME
# ================================

df = pd.DataFrame(
    [x.strip().split(",") for x in dirty_data[1:]],
    columns=dirty_data[0].strip().split(",")
)

# ===============================
# EXPLORE DATA
# ================================
# print(df.head())
# print(df.info())
# print(df.describe())
# print(df[df.isna()])


# ================================
# CLEAN DATA
# ================================

# Replace empty / whitespace with NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

# Convert numeric columns
numeric_cols = [
    "vehicle_age_years", "mileage_km", "engine_hours",
    "last_service_km_ago", "oil_quality_pct",
    "avg_trip_length_km", "cleanliness_score",
    "driver_satisfaction_score"
]

# Convert numeric columns, coercing errors to NaN
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert target
# The target column "breakdown_next_30_days" should be binary (0/1). We will convert it to numeric, coercing errors to NaN.
df["breakdown_next_30_days"] = pd.to_numeric(df["breakdown_next_30_days"], errors="coerce")

# ================================
# HANDLE MISSING VALUES
# ================================

# Fill numeric NaNs with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical NaNs with "unknown"
categorical_cols = ["vehicle_brand", "fuel_type", "tyre_type", "weather_exposure"]

for col in categorical_cols:
    df[col] = df[col].fillna("unknown")

# ================================
# ENCODE CATEGORICAL VARIABLES
# ================================
df = pd.get_dummies(df, columns=categorical_cols)

# ================================
# SPLIT FEATURES / TARGET
# ================================
from sklearn.model_selection import train_test_split

X = df.drop(columns=["id", "breakdown_next_30_days"])
y = df["breakdown_next_30_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# TRAIN MODEL
# ================================
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# ================================
# EVALUATE
# ================================
accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

# ================================
# PREDICT
# ================================
predictions = model.predict(X)

# ================================
# OUTPUT FILE (KAGGLE FORMAT)
# ================================
output = pd.DataFrame({
    "id": df["id"].astype(int),
    "breakdown_next_30_days": predictions.astype(int)
})

output.to_csv("submission.csv", index=False)

       id vehicle_brand vehicle_age_years mileage_km engine_hours  \
0     NaN           NaN               NaN        NaN          NaN   
1     NaN           NaN               NaN        NaN          NaN   
2     NaN           NaN               NaN        NaN          NaN   
3     NaN           NaN               NaN        NaN          NaN   
4     NaN           NaN               NaN        NaN          NaN   
...   ...           ...               ...        ...          ...   
1045  NaN           NaN               NaN        NaN          NaN   
1046  NaN           NaN               NaN        NaN          NaN   
1047  NaN           NaN               NaN        NaN          NaN   
1048  NaN           NaN               NaN        NaN          NaN   
1049  NaN           NaN               NaN        NaN          NaN   

     last_service_km_ago oil_quality_pct avg_trip_length_km weather_exposure  \
0                    NaN             NaN                NaN              NaN   
1          